## Import Packages

In [ ]:
!pip install dask

In [ ]:
import numpy as np
import pandas as pd
import dask.dataframe as dd
import matplotlib.pyplot as plt
import seaborn as sns

## Load Data

In [ ]:
# paths for the three dfs

df_jan_path = "/kaggle/input/datasets/elemento/nyc-yellow-taxi-trip-data/yellow_tripdata_2016-01.csv"
df_feb_path = "/kaggle/input/datasets/elemento/nyc-yellow-taxi-trip-data/yellow_tripdata_2016-02.csv"
df_mar_path = "/kaggle/input/datasets/elemento/nyc-yellow-taxi-trip-data/yellow_tripdata_2016-03.csv"


- Based on the exploratory data analysis (EDA) of NYC Yellow Taxi trips, we identified outliers in the data. I selected specific columns that are useful for demand prediction, and these columns also contain outliers. Therefore, I need to remove the outliers from these specific columns.
- Columns are 'trip_distance', 'pickup_longitude',
       'pickup_latitude','dropoff_longitude', 'dropoff_latitude', 'fare_amount'

In [ ]:
# load the dataframes

df_jan = dd.read_csv(df_jan_path, assume_missing=True, usecols= ['trip_distance', 'tpep_pickup_datetime', 'pickup_longitude',
       'pickup_latitude','dropoff_longitude', 'dropoff_latitude', 'fare_amount'], parse_dates=["tpep_pickup_datetime"])

df_feb = dd.read_csv(df_feb_path, assume_missing=True, usecols= ['trip_distance', 'tpep_pickup_datetime', 'pickup_longitude',
       'pickup_latitude','dropoff_longitude', 'dropoff_latitude', 'fare_amount'], parse_dates=["tpep_pickup_datetime"])


df_mar = dd.read_csv(df_mar_path, assume_missing=True, usecols= ['trip_distance', 'tpep_pickup_datetime', 'pickup_longitude',
       'pickup_latitude','dropoff_longitude', 'dropoff_latitude', 'fare_amount'], parse_dates=["tpep_pickup_datetime"])

In [ ]:
# concat the three dataframes as one

df_final = dd.concat([df_jan, df_feb, df_mar], axis=0) # axis =0 means It works vertically (up and down).

In [ ]:
df_final.head()

### **New york bounding box:**
min_latitude = 40.60
max_latitude = 40.85
min_longitude = -74.05
max_longitude = -73.70

In [ ]:
# set the values of coordinates

min_latitude = 40.60
max_latitude = 40.85
min_longitude = -74.05
max_longitude = -73.70

In [ ]:
# fare amount column
fare_amount = df_final["fare_amount"].compute()

# trip distance column
trip_distance = df_final["trip_distance"].compute()

In [ ]:
fare_amount.shape[0]/10000000

In [ ]:
## Percentile of fare amount
percentiles=np.arange(0.991,1,0.001)
fare_amount.quantile(percentiles)

In [ ]:
max_fare_amount_val = fare_amount.quantile(percentiles).iloc[-2].item()
min_fare_amount_val = 0.50

print(min_fare_amount_val)
print(max_fare_amount_val)

In [ ]:
trip_distance.quantile(percentiles)

In [ ]:
# percentile values for trip_distance

min_trip_distance_val = 0.25
max_trip_distance_val = trip_distance.quantile(percentiles).iloc[-2].item()

print(min_trip_distance_val)
print(max_trip_distance_val)

## Remove Outlier from the location data

In [ ]:
# select data points within the given ranges

df_final = df_final.loc[(df_final["pickup_latitude"].between(min_latitude, max_latitude, inclusive="both")) & 
(df_final["pickup_longitude"].between(min_longitude, max_longitude, inclusive="both")) & 
(df_final["dropoff_latitude"].between(min_latitude, max_latitude, inclusive="both")) & 
(df_final["dropoff_longitude"].between(min_longitude, max_longitude, inclusive="both")), :]

## Remove Outliers from the Fare Amount data and Distance

In [ ]:
df_final = df_final.loc[(df_final["fare_amount"].between(min_fare_amount_val,max_fare_amount_val,inclusive="both")) & 
(df_final["trip_distance"].between(min_trip_distance_val,max_trip_distance_val,inclusive="both"))]

## Remove Outliers from the Distance data

In [ ]:
df_final = df_final.loc[(df_final["fare_amount"].between(min_fare_amount_val,max_fare_amount_val,inclusive="both")) & 
(df_final["trip_distance"].between(min_trip_distance_val,max_trip_distance_val,inclusive="both"))]

## Save After Removing Outlier

In [ ]:
# save the pickup coordinates dataset
import os

dir_path = "/kaggle/working/data/interim/"
file_path = os.path.join(dir_path, "location_data.csv")
pickup_coord_dataset = df_final.loc[:,['tpep_pickup_datetime','pickup_latitude','pickup_longitude']]

In [ ]:
# form the dataset

pickup_coord_dataset = df_final.loc[:,['tpep_pickup_datetime','pickup_latitude','pickup_longitude']].compute()

print("Shape of the dataset is ", pickup_coord_dataset.shape)

In [ ]:
import os
os.makedirs(dir_path, exist_ok=True)

In [ ]:
pickup_coord_dataset.to_csv(file_path, index=False)

## Making The Regions 

In [1]:
import pandas as pd
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler

In [2]:
df_reader=pd.read_csv("/kaggle/working/data/interim/location_data.csv",chunksize=100000, usecols=["pickup_latitude","pickup_longitude"])

In [3]:
# train the standard scaler

scaler = StandardScaler()

for chunk in df_reader:
    # fit the scaler
    scaler.partial_fit(chunk)

In [4]:
# 1. Re-initialize the iterator (since cell 73 consumed it completely)
df_reader = pd.read_csv("/kaggle/working/data/interim/location_data.csv", chunksize=100000, usecols=["pickup_latitude", "pickup_longitude"])

# 2. Train the MiniBatchKMeans model
mini_batch = MiniBatchKMeans(n_clusters=30, n_init=10, random_state=42)
for chunk in df_reader:
    scaled_chunk = scaler.transform(chunk)
    mini_batch.partial_fit(scaled_chunk)

# 3. Access centroids
mini_batch.cluster_centers_

array([[ 1.94149572,  0.6614905 ],
       [-0.13980742, -0.07596928],
       [-1.99721801,  1.44528196],
       [-3.83635934,  5.15118517],
       [-1.16343866, -0.81227839],
       [ 0.42137614, -0.13488625],
       [ 0.72209007,  2.8610836 ],
       [-0.52924948, -0.39015448],
       [ 1.07003845,  0.56520318],
       [-2.24451716, -0.3303071 ],
       [-1.00166487, -0.4036074 ],
       [ 1.16185672, -0.10175947],
       [-0.08902033, -0.55920139],
       [ 0.31820801,  1.59046972],
       [-0.04552948, -0.25372558],
       [ 0.20846098, -0.35389222],
       [-1.31780812,  0.52314378],
       [ 2.79102906,  0.81932304],
       [ 0.28712745,  0.12935608],
       [ 0.73099844, -0.27639436],
       [-1.53772972, -1.0082658 ],
       [ 0.67692302,  0.39844824],
       [-0.76412025, -0.74701446],
       [ 1.62394033,  0.1378545 ],
       [-3.07963657, -0.43385234],
       [ 0.3506753 , -0.54018812],
       [-0.57100818, -0.17842182],
       [-0.38830582, -0.76072368],
       [-2.66479788,

In [5]:
scaler.inverse_transform(mini_batch.cluster_centers_)

array([[ 40.80392392, -73.94975046],
       [ 40.74726528, -73.97685385],
       [ 40.69670159, -73.92094427],
       [ 40.64663525, -73.78474354],
       [ 40.7193993 , -74.00391496],
       [ 40.7625422 , -73.97901919],
       [ 40.77072843, -73.8689102 ],
       [ 40.73666362, -73.9884009 ],
       [ 40.78020052, -73.95328925],
       [ 40.68996945, -73.98620137],
       [ 40.72380321, -73.98889533],
       [ 40.78270006, -73.9778017 ],
       [ 40.74864784, -73.99461378],
       [ 40.75973368, -73.91560827],
       [ 40.74983178, -73.98338682],
       [ 40.75674608, -73.98706818],
       [ 40.71519695, -73.95483503],
       [ 40.82705049, -73.94394974],
       [ 40.75888759, -73.96930766],
       [ 40.77097094, -73.98421995],
       [ 40.70921009, -74.01111796],
       [ 40.76949887, -73.95941789],
       [ 40.73026981, -74.00151635],
       [ 40.79527921, -73.96899532],
       [ 40.66723527, -73.9900069 ],
       [ 40.76061753, -73.993915  ],
       [ 40.73552684, -73.98061923],
 

In [6]:
final_df=pd.read_csv("/kaggle/working/data/interim/location_data.csv")

In [7]:
final_df.head()

,tpep_pickup_datetime,pickup_latitude,pickup_longitude
0,2016-01-01 00:00:00,40.734695,-73.990372
1,2016-01-01 00:00:00,40.729912,-73.980782
2,2016-01-01 00:00:00,40.679565,-73.984550
3,2016-01-01 00:00:00,40.718990,-73.993469
4,2016-01-01 00:00:00,40.781330,-73.960625


In [8]:
# prediction 
scaled_location_subset = scaler.transform(final_df.iloc[:, 1:])


scaled_location_subset

# get the cluster predictions

cluster_predictions = mini_batch.predict(scaled_location_subset)

cluster_predictions.shape

(33234199,)

In [9]:
# save the cluster predictions in data

# Save the cluster predictions directly into your Pandas DataFrame
final_df['region'] = cluster_predictions
time_series_data = final_df

save_path = "/kaggle/working/data/interim/time_series_location.csv"

time_series_data.to_csv(save_path, index=False)

In [ ]:
# Save the cluster predictions directly into your Pandas DataFrame
final_df['region'] = cluster_predictions
time_series_data = final_df.drop(columns=["pickup_latitude","pickup_longitude"])

save_path = "/kaggle/working/data/interim/time_series_location.csv"

time_series_data.to_csv(save_path, index=False)

In [ ]:
# import shutil 

# shutil.rmtree("/kaggle/working/data/interim/time_series.csv")

* Time Series Data

In [ ]:
time_series_data=pd.read_csv("/kaggle/working/data/interim/time_series.csv")

In [ ]:
time_series_data.head()

In [ ]:
time_series_data['tpep_pickup_datetime'] = pd.to_datetime(time_series_data['tpep_pickup_datetime'])

In [ ]:
time_series_data.set_index('tpep_pickup_datetime', inplace=True)

time_series_data

In [ ]:
region_group=time_series_data.groupby("region")

In [ ]:
time_series_data.isna().sum()

In [ ]:
# resample the time series in 15 minute intervals

resampled_data = (
    region_group['region']
    .resample("15min")
    .count()
)

resampled_data

In [ ]:

resampled_data.name = "total_pickups"

In [ ]:
resampled_data = resampled_data.reset_index(level=0)

resampled_data

In [ ]:

# zeros in the data

(resampled_data['total_pickups'] == 0).sum()

In [ ]:
epsilon_val = 10

resampled_data.replace({'total_pickups': {0 : epsilon_val}}, inplace=True)

In [ ]:

(resampled_data['total_pickups'] == 0).sum()

In [ ]:

from sklearn.metrics import mean_absolute_percentage_error

In [ ]:
window_values = list(range(3,11,1))
window_values

In [ ]:
def calculate_best_window_value(windows):
    for window in windows:
        ind = window - 1
        y_pred = resampled_data['total_pickups'].rolling(window=window).mean().values[ind:]
        y = resampled_data['total_pickups'].values[ind:]
        error = mean_absolute_percentage_error(y, y_pred)
        print(f"For window value {window}, the MAPE is {error:.2f}")

In [ ]:

calculate_best_window_value(window_values)

In [ ]:
resampled_data['total_pickups'].ewm(alpha=0.9).mean()

In [ ]:

smoothing_values = np.arange(0.2,1,0.1)
smoothing_values

In [ ]:

def calculate_best_smoothing_value(values):
    y = resampled_data['total_pickups'].values
    for value in values:
        y_pred = resampled_data['total_pickups'].ewm(alpha=value).mean()
        error = mean_absolute_percentage_error(y, y_pred)
        print(f"For smoothing value {value:.1f}, the MAPE is {error:.2f}")

In [ ]:

calculate_best_smoothing_value(smoothing_values)

In [ ]:

# dataset with pickup smoothing applied (shifted by 1 to avoid data leakage)

resampled_data["avg_pickups"] = resampled_data['total_pickups'].ewm(alpha=0.4).mean().shift(1).round()

resampled_data

In [ ]:

# save the resampled data

resampled_data_save_path = "/kaggle/working/data/interim/final_data.csv"

resampled_data.to_csv(resampled_data_save_path, index=True)

## Model Building

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# load the data

data_path = "/kaggle/working/data/interim/final_data.csv"

df = pd.read_csv(data_path, parse_dates=["tpep_pickup_datetime"])

In [ ]:
# shape of the data

df.shape

In [ ]:

# extract the day of week information
df["day_of_week"] = df["tpep_pickup_datetime"].dt.day_of_week

# extract the month information
df["month"] = df["tpep_pickup_datetime"].dt.month

In [ ]:
# set the datetime column as index

df.set_index("tpep_pickup_datetime", inplace=True)
df

In [ ]:
# create the region grouper

region_grp = df.groupby("region")

region_grp.

In [ ]:
# shifting periods

periods = list(range(1,5))

periods

In [ ]:
# generate the lag features

lag_features = region_grp["total_pickups"].shift(periods)

lag_features

In [ ]:
# merge them with the original df

data = pd.concat([lag_features,df],axis=1)

data

In [ ]:
print("The shape of the df before merger ", df.shape)
print("The shape of the df after merger ", data.shape)


In [ ]:
# rows having missing values

data.isna().any(axis=1).sum()


In [ ]:
# drop the missing values

data.dropna(inplace=True)
data.isna().any(axis=1).sum()

In [ ]:
mapper = {name:f"lag_{ind+1}" for ind, name in enumerate(data.columns[0:4])}

mapper

In [ ]:
# replace the column names

data = data.rename(columns=mapper)

In [ ]:
# number of rows in each month

data['month'].value_counts()

In [ ]:
data.loc[data["month"].isin([1,2]),"lag_1":"day_of_week"]

In [ ]:
# split the data

trainset = data.loc[data["month"].isin([1,2]),"lag_1":"day_of_week"]

testset = data.loc[data["month"].isin([3]),"lag_1":"day_of_week"]
trainset 

In [ ]:
# save the train and test data

train_data_save_path = "/kaggle/working/data/interim/train.csv"

test_data_save_path = "/kaggle/working/data/interim/test.csv"

trainset.to_csv(train_data_save_path, index=True)
testset.to_csv(test_data_save_path, index=True)

In [ ]:
# make X_train and y_train

X_train = trainset.drop(columns=["total_pickups"])

y_train = trainset["total_pickups"]

In [ ]:
X_train.shape

In [ ]:
y_train.shape

In [ ]:
# make X_test and y_test

X_test = testset.drop(columns=["total_pickups"])

y_test = testset["total_pickups"]

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_percentage_error

In [ ]:
from sklearn import set_config

set_config(transform_output="pandas")

In [ ]:

# encode the data

encoder = ColumnTransformer([
    ("ohe", OneHotEncoder(drop="first",sparse_output=False), ["region","day_of_week"])
], remainder="passthrough", n_jobs=-1,force_int_remainder_cols=False)

In [ ]:
# encode the train and test data

X_train_encoded = encoder.fit_transform(X_train)
X_test_encoded = encoder.transform(X_test)

In [ ]:
# encode the train and test data

X_train_encoded = encoder.fit_transform(X_train)
X_test_encoded = encoder.transform(X_test)

In [ ]:
# train the model

lr = LinearRegression()

# fit on the training data
lr.fit(X_train_encoded, y_train)

In [ ]:
# make predictions on the train data

y_pred_train = lr.predict(X_train_encoded)
# make predictions on the test data

y_pred_test = lr.predict(X_test_encoded)

In [ ]:
# evaluate the baseline model

train_mape = mean_absolute_percentage_error(y_train, y_pred_train)

test_mape = mean_absolute_percentage_error(y_test, y_pred_test)

In [ ]:
test_mape

In [ ]:
print(f"MAPE on trainset is {(train_mape * 100):.2f}%")
print(f"MAPE on testset is {(test_mape * 100):.2f}%")

## Hyper Parmeter tuning

In [ ]:
!pip install mlflow

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.svm import SVR

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

import mlflow

In [ ]:
mlflow.set_tracking_uri("https://dagshub.com/iamdebasishdas123/Uber-Demand-Prediction.mlflow")

In [ ]:
!pip install dagshub

In [ ]:
import dagshub
dagshub.init(repo_owner='iamdebasishdas123', repo_name='Uber-Demand-Prediction', mlflow=True)

In [ ]:
# load the training and test data

train_data_path = "/kaggle/working/data/interim/train.csv"

test_data_path = "/kaggle/working/data/interim/test.csv"

train_df = pd.read_csv(train_data_path, parse_dates=["tpep_pickup_datetime"]).set_index("tpep_pickup_datetime")

test_df = pd.read_csv(test_data_path, parse_dates=["tpep_pickup_datetime"]).set_index("tpep_pickup_datetime")

train_df

In [ ]:
# missing value in training data

train_df.isna().sum()

In [ ]:
# missing values in the test data

test_df.isna().sum()

In [ ]:
# make X_train and y_train

X_train = train_df.drop(columns=["total_pickups"])

y_train = train_df["total_pickups"]

In [ ]:

X_test.head()

In [ ]:
from sklearn import set_config

set_config(transform_output="pandas")

# encode the data

encoder = ColumnTransformer([
    ("ohe", OneHotEncoder(drop="first",sparse_output=False), ["region","day_of_week"])
], remainder="passthrough", n_jobs=-1,force_int_remainder_cols=False)

In [ ]:
encoder

In [ ]:
# encode the train and test data

X_train_encoded = encoder.fit_transform(X_train)
X_test_encoded = encoder.transform(X_test)

In [ ]:
import optuna
import tqdm 

In [ ]:
# set the experiment

mlflow.set_experiment("Model Selection")

*OPtuna*
Optuna works by framing hyperparameter optimization as an iterative search problem where a central orchestrator decides which parameter combinations to test based on past performance.

To build an Optuna optimization script, you only need four core components:

- An Objective Function: A standard Python function that accepts an Optuna trial object as its input.
It must contain your data loading, model initialization, training loop, and evaluation step.
- Suggested Hyperparameters: Inside your objective function, you must replace your fixed numbers with Optuna suggestion methods.Examples: trial.suggest_float(), trial.suggest_int(), or trial.suggest_categorical().
- A Return Metric: Your objective function must return a single scalar value (a float) at the very end.This is the metric Optuna will look at to judge whether the trial was a success or a failure.
- A Study Object: Generated via optuna.create_study().You must explicitly tell it whether to maximize your return metric (e.g., accuracy, F1-score) or minimize it (e.g., MSE loss, cross-entropy).

In [ ]:
def objective(trial):
    # start the child run
    with mlflow.start_run(nested=True) as child:
        
        # model name search space
        list_of_models = ["LR", "RF", "GBR", "XGBR"]
        model_name = trial.suggest_categorical("model_name", list_of_models)
    
        if model_name == "LR":
            model = LinearRegression()
    
        elif model_name == "RF":
            n_estimators_rf = trial.suggest_int("n_estimators_rf",10,100,step=10)
            max_depth_rf = trial.suggest_int("max_depth_rf",3,10)
            model = RandomForestRegressor(n_estimators=n_estimators_rf, 
                                          max_depth=max_depth_rf, 
                                          random_state=42, n_jobs=-1)
    
        elif model_name == "GBR":
            n_estimators_gb = trial.suggest_int("n_estimators_gb",10,100,step=10)
            learning_rate_gb = trial.suggest_float("learning_rate_gb",1e-4,1e-1, log=True)
            model = GradientBoostingRegressor(n_estimators=n_estimators_gb, 
                                              learning_rate=learning_rate_gb,
                                             random_state=42)
    
        elif model_name == "XGBR":
            n_estimators_xgb = trial.suggest_int("n_estimators_xgb",10,100,step=10)
            learning_rate_xgb = trial.suggest_float("learning_rate_xgb",1e-4,1e-1, log=True)
            max_depth_xgb = trial.suggest_int("max_depth_xgb",3,10)
            model = XGBRegressor(n_estimators=n_estimators_xgb,
                                learning_rate=learning_rate_xgb,
                                max_depth=max_depth_xgb)
    
        # log the model name
        mlflow.log_param("model_name",model_name)
        
        # log the model parameters
        mlflow.log_params(model.get_params())
        
        # fit on the data
        model.fit(X_train_encoded,y_train)
    
        # get the predictions
        y_pred = model.predict(X_test_encoded)
    
        # calculate the loss
        loss = mean_absolute_percentage_error(y_test, y_pred)
    
        # log the metric
        mlflow.log_metric("MAPE",loss)
        return loss

In [ ]:
# optimize the objective function

with mlflow.start_run(run_name="best_model", nested=True) as parent:

    # create a study object
    study = optuna.create_study(study_name="model_selection", direction="minimize")
    # optimize the objective function
    study.optimize(func=objective, n_trials=50, n_jobs=-1)
    
    # log the best parameters
    mlflow.log_params(study.best_params)
    # log the best error value
    mlflow.log_metric("Best_MAPE", study.best_value)

In [ ]:
# best value

study.best_value

In [ ]:
# best parameters

study.best_params

In [ ]:
# model value counts

study.trials_dataframe()['params_model_name'].value_counts()

## Make Plot data csv

In [10]:
import pandas as pd 
df_location=pd.read_csv("/kaggle/working/data/interim/time_series_location.csv")

In [11]:
df_location.head()

,tpep_pickup_datetime,pickup_latitude,pickup_longitude,region
0,2016-01-01 00:00:00,40.734695,-73.990372,7
1,2016-01-01 00:00:00,40.729912,-73.980782,26
2,2016-01-01 00:00:00,40.679565,-73.984550,9
3,2016-01-01 00:00:00,40.718990,-73.993469,10
4,2016-01-01 00:00:00,40.781330,-73.960625,8


In [13]:
# Sample 20 random data points per region
df_location = (
    df_location.groupby("region", group_keys=False)
    .apply(lambda x: x.sample(n=200, random_state=42))
    .reset_index(drop=True)
)

# Verify shape (should be (600, column_count))
print("Sampled dataset shape:", df_location.shape)

# Save the 600-point dataset
# df_sampled.to_csv("/kaggle/working/data/interim/sampled_600_points.csv", index=False)

Sampled dataset shape: (6000, 4)


/tmp/ipykernel_58/2458524797.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=200, random_state=42))


In [16]:
df_location.to_csv("/kaggle/working/data/interim/sampled_6000_points.csv", index=False)